In [9]:
import math
class Value:
    def __init__(self, data, _children=(), _op=''):
        # 存储真实的，向前计算的数值
        self.data = data
        # 损失函数对当前变量的导数，初始化为0
        self.grad = 0
        # 反向传播函数（默认无）
        self._backward = lambda: None
        # 记录前驱节点，依靠什么变量生成的,让节点知道自己的父变量
        self._prev = set(_children)
        # 记录是什么运算生成的（+， *， ReLu...)
        self._op = _op


    def __add__(self, other):
        #支持类型兼容，输入原生类型，可以自动包装成Value对象
        #进而支持 X + int这种运算
        other = other if isinstance(other, Value) else Value(other)
        # 对Value对象内部进行data相加，最后返回一个Value对象
        #包含self,other2个父变量，+的运算标识
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # a + b = c; L = f(a, b, c)
            # dL/da = (dL/dc)*(dc/da) = out.grad
            # 选用+=，一个变量的总梯度等于所有流经它的路径梯度的总和。
            self.grad += out.grad
            other.grad += out.grad
        #闭包，将_backward绑定到out上，其父节点self, other被直接绑定
        #当反向传播时可以通过out._backward()，作用到两个父节点上
        out._backward = _backward

        return out
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # c = a * b; L = f(a, b, c)
            # dL/da = (dL/dc)*(dc/da) = out.grad * b.grad
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        #只支持整数和浮点数运算，不支持变量
        assert isinstance(other, (int, float))
        # 因为支持的是纯数字，不是Value包装。不用参与反向传播
        #所以只记录self，表示运算处就加上other，方便查看
        out = Value(self.data**other, (self,),f'**{other}')

        def _backward():
        # a**b = c;L = f(a, c)
        # dL/da = (dL/dc) * (dc/da)
        # dL/da = out.grad * b * a**(b-1)
            self.grad += other * self.data**(other-1) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        # if x <= 0, 0 = relu(x);
        # if x > 0, x = relu(x)
        #当输入数据小于0，直接Value(0),否则直接value包装
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out
        
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2)*out.grad
        out._backward = _backward    
        return out
        
    #定义反向传播核心函数，backward串联其所有的_backward
    # backward实现拓扑排序，所有子节点全部算出后才能进入下一层节点计算梯度
    # 倒序执行链式法则，从Loss到要求的关键参数的梯度
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                # append是后序变量，在最后处添加新变量
                topo.append(v)
        build_topo(self)
        # 设置out.grad = 1
        self.grad = 1
        #倒序处理
        for v in reversed(topo):
            v._backward()
        
    #支持取负（-self)
    def __neg__(self):
        return self * -1

    #解决2 + a时， 2.__add__(a)报错问题
    def __radd__(self, other):
    # 2 + a时，会先尝试 2.__add__(a)，错误，尝试a.__radd__(2)
    #返回 a + 2 调用 a.__add__(2)，成功
        return self + other

    #以下同样的思路
    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other): 
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    #重写Value表示，打印出的是Value(data=3.0, grad=0.0)
    #而不是：<__main__.Value object at 0x7f8b1c0b3a90>
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"





#测试demo

if __name__ == "__main__":
    print("========== 1. 定义输入与可训练权重 ==========")
    x1 = Value(2.0)  # 输入特征 1
    x2 = Value(0.0)  # 输入特征 2
    w1 = Value(-3.0)  # 权重 1
    w2 = Value(1.0)  # 权重 2
    b = Value(6.8813735870195432)  # 偏置

    print(f"初始权重 w1: {w1}")
    print(f"初始偏置 b:  {b}")

    print("\n========== 2. 前向传播（混合各种运算）==========")
    # 模拟神经元线性组合: x1*w1 + x2*w2 + b
    # 顺便测试之前写的: 乘法、加法、反向加法(数字在左)、减法、除法等
    x1w1 = x1 * w1  # 2.0 * -3.0 = -6.0
    x2w2 = x2 * w2  # 0.0 * 1.0 = 0.0
    x1w1_x2w2 = x1w1 + x2w2  # -6.0
    n = 2 + x1w1_x2w2 + b - 2  # 测试反向加减: 结果依然是 -6.0 + 6.8814 = 0.8814

    # 激活函数
    out = n.relu()

    # 假设目标真实值 y_true = 0.0，计算均方误差损失 Loss = (out - 0.0)^2
    loss = (out - 0.0) ** 2
    print(f"前向输出 n:    {n}")
    print(f"ReLU激活后 out: {out}")
    print(f"最终损失 Loss:  {loss}")

    print("\n========== 3. 反向传播（一键求导）==========")
    loss.backward()

    print("反向传播完成后的梯度：")
    print(f"Loss 的梯度: {loss}")
    print(f"w1   的梯度: {w1}")  # dLoss/dw1
    print(f"x1   的梯度: {x1}")  # dLoss/dx1
    print(f"b    的梯度: {b}")  # dLoss/db
    print(f"w2   的梯度: {w2}")
    
    print("\n========== 4. 模拟一次梯度下降更新 ==========")
    learning_rate = 0.01

    # 参数更新规则: w = w - lr * w.grad
    w1.data -= learning_rate * w1.grad
    b.data -= learning_rate * b.grad

    print(f"更新一步后的权重 w1: {w1.data:.4f}")
    print(f"更新一步后的偏置 b:  {b.data:.4f}")


========== 1. 定义输入与可训练权重 ==========
初始权重 w1: Value(data=-3.0, grad=0)
初始偏置 b:  Value(data=6.881373587019543, grad=0)

========== 2. 前向传播（混合各种运算）==========
前向输出 n:    Value(data=0.8813735870195432, grad=0)
ReLU激活后 out: Value(data=0.8813735870195432, grad=0)
最终损失 Loss:  Value(data=0.7768193998956963, grad=0)

========== 3. 反向传播（一键求导）==========
反向传播完成后的梯度：
Loss 的梯度: Value(data=0.7768193998956963, grad=1)
w1   的梯度: Value(data=-3.0, grad=3.5254943480781726)
x1   的梯度: Value(data=2.0, grad=-5.288241522117259)
b    的梯度: Value(data=6.881373587019543, grad=1.7627471740390863)
w2   的梯度: Value(data=1.0, grad=0.0)

========== 4. 模拟一次梯度下降更新 ==========
更新一步后的权重 w1: -3.0353
更新一步后的偏置 b:  6.8637
